# Maybank (1155.KL) Trading Signal Prediction with XGBoost
### Using FBM KLCI (Top 30 Malaysian companies index, which includes Maybank) as a market-context feature

This notebook builds a **3-class classifier** that predicts a next-day trading action for **Maybank (1155.KL)**:

| Class | Meaning | Trigger |
|---|---|---|
| `0` | **Hold** | next-day return stays inside a neutral band |
| `1` | **Buy**  | next-day return is meaningfully positive |
| `2` | **Sell** | next-day return is meaningfully negative |

using:
- Maybank's own OHLCV-derived technical features (momentum, trend, volatility, volume)
- FBM KLCI (`^KLSE`) index-derived features, since Maybank is a large constituent of the index and broad market direction often leads/confirms individual stock moves

**Structure of this notebook:**
1. Data Cleaning
2. Time-Series Data Split (Train / Valid / Test = 7:2:1)
3. Feature Engineering
4. XGBoost Hyperparameter Tuning (with early stopping)
5. Baseline Comparison
6. Full Model Evaluation Results
7. Test (Production) Score
8. Mock Trading Simulation

> **Note:** This notebook downloads live data via `yfinance`, so it must be run in an environment with internet access (e.g. locally or on Google Colab).

## 0. Setup & Installs

Install/import all required libraries. `ta` is used for technical-analysis indicators, `xgboost` for the classifier.

In [ ]:
# Dependencies and requirements for the notebook
# If running in a fresh environment (e.g. Colab), uncomment the line below:


In [ ]:


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf
import ta

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
RANDOM_STATE = 42

In [ ]:
# --- Config ---
MAYBANK_TICKER = "1155.KL"   # Maybank on Bursa Malaysia
KLCI_TICKER    = "^KLSE"     # FTSE Bursa Malaysia KLCI (top 30 companies, incl. Maybank)

START_DATE = "2010-01-01"
END_DATE   = None  # None = up to today

print(f"Maybank ticker : {MAYBANK_TICKER}")
print(f"FBM KLCI ticker: {KLCI_TICKER}")

## Download Raw Data

We pull daily OHLCV data for both Maybank and the FBM KLCI index. The KLCI series will later be merged in as *contextual/market* features — Maybank is one of the index's largest constituents, so the index's own momentum/trend can carry predictive signal for the individual stock.

In [ ]:
maybank_raw = yf.download(MAYBANK_TICKER, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)
klci_raw    = yf.download(KLCI_TICKER, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)

# yfinance sometimes returns MultiIndex columns for a single ticker - flatten if needed
if isinstance(maybank_raw.columns, pd.MultiIndex):
    maybank_raw.columns = maybank_raw.columns.get_level_values(0)
if isinstance(klci_raw.columns, pd.MultiIndex):
    klci_raw.columns = klci_raw.columns.get_level_values(0)

print("Maybank shape:", maybank_raw.shape)
print("KLCI shape   :", klci_raw.shape)
maybank_raw.tail()

## 1. Data Cleaning

Requirement: **drop rows that have null values in OHLCV (Open, High, Low, Close, Volume)**, for both Maybank and the FBM KLCI series, before any feature engineering happens. We also align both series on the trading calendar (inner join on date), since holidays can differ slightly and we need matched dates when merging KLCI features onto Maybank rows.

In [ ]:
def clean_ohlcv(df, name):
    ohlcv_cols = ["Open", "High", "Low", "Close", "Volume"]
    before = len(df)
    df = df.copy()

    # Keep only rows where all OHLCV columns are present and non-null
    df = df.dropna(subset=ohlcv_cols)

    # Defensive: drop any rows where volume is 0 (non-trading / bad data) which can break pct_change features
    df = df[df["Volume"] > 0]

    after = len(df)
    print(f"[{name}] Dropped {before - after} rows with null/invalid OHLCV out of {before} total rows.")
    return df

maybank_clean = clean_ohlcv(maybank_raw, "Maybank")
klci_clean    = clean_ohlcv(klci_raw, "FBM KLCI")

# Align both series on shared trading dates (inner join) so KLCI features line up 1-to-1 with Maybank rows
common_dates = maybank_clean.index.intersection(klci_clean.index)
maybank_clean = maybank_clean.loc[common_dates].sort_index()
klci_clean    = klci_clean.loc[common_dates].sort_index()

print(f"\nAligned shape (both series): {maybank_clean.shape}")
assert maybank_clean.isna().sum().sum() == 0, "Nulls remain in Maybank OHLCV!"
assert klci_clean.isna().sum().sum() == 0, "Nulls remain in KLCI OHLCV!"
print("No nulls remain in either OHLCV dataset. ✅")

## 2. Time-Series Data Split (Train : Valid : Test = 7 : 2 : 1)

Because this is time-series data, we **never shuffle**. We split chronologically:

- **Train (70%)** — used to fit the model
- **Valid (20%)** — used for hyperparameter tuning / early stopping
- **Test (10%)** — held out completely, touched only once at the end, to simulate genuine "production" performance on unseen future data

We compute the split **cutoff dates** here on the cleaned, aligned date index. The actual feature matrix (built in Section 3) will be sliced using these same cutoff dates, so there is no leakage of future information into earlier splits (e.g. no rolling-window feature is allowed to peek across the boundary in a way that uses test labels).

In [ ]:
n = len(maybank_clean)
train_end_idx = int(n * 0.7)
valid_end_idx = int(n * 0.9)   # 0.7 + 0.2

train_dates = maybank_clean.index[:train_end_idx]
valid_dates = maybank_clean.index[train_end_idx:valid_end_idx]
test_dates  = maybank_clean.index[valid_end_idx:]

print(f"Total rows : {n}")
print(f"Train      : {len(train_dates):>5} rows  ({train_dates.min().date()} -> {train_dates.max().date()})")
print(f"Valid      : {len(valid_dates):>5} rows  ({valid_dates.min().date()} -> {valid_dates.max().date()})")
print(f"Test (prod): {len(test_dates):>5} rows  ({test_dates.min().date()} -> {test_dates.max().date()})")
print(f"\nRatio check -> train: {len(train_dates)/n:.2%}, valid: {len(valid_dates)/n:.2%}, test: {len(test_dates)/n:.2%}")

## 3. Feature Engineering

We build two feature blocks:

1. **Maybank technical features** — momentum (RSI, MACD), trend/moving-average ratios, ADX, rolling return mean/std, and volume features (exactly as specified).
2. **FBM KLCI market-context features** — the *same* style of technical features computed on the index itself, prefixed `klci_`, then merged onto the Maybank rows by date. Since Maybank is a top constituent of the KLCI, the index's own trend/momentum acts as a broad market signal that can help/confirm Maybank's own direction.

All rolling/indicator calculations are done on the **full continuous price history** (not per-split) since these are standard causal (backward-looking only) technical indicators — each value at time *t* only uses data up to and including *t*, so there is no leakage across the train/valid/test boundaries we defined above.

In [ ]:
def build_technical_features(df, prefix=""):
    """Builds the requested technical-analysis feature set on a single OHLCV dataframe.
    Returns a DataFrame of features only (same index as df), optionally prefixed
    (used to distinguish KLCI features from Maybank features after merging).
    """
    close  = df["Close"]
    high   = df["High"]
    low    = df["Low"]
    volume = df["Volume"]

    feat = pd.DataFrame(index=df.index)

    # --- Momentum ---
    feat["rsi_14"] = ta.momentum.RSIIndicator(close, window=14).rsi()

    macd = ta.trend.MACD(close)
    feat["macd"] = macd.macd()
    feat["macd_signal"] = macd.macd_signal()
    feat["macd_diff"] = macd.macd_diff()

    # --- Trend / moving averages (ratios, not raw price, for stationarity) ---
    sma_5  = ta.trend.SMAIndicator(close, window=5).sma_indicator()
    sma_10 = ta.trend.SMAIndicator(close, window=10).sma_indicator()
    sma_20 = ta.trend.SMAIndicator(close, window=20).sma_indicator()
    sma_50 = ta.trend.SMAIndicator(close, window=50).sma_indicator()

    feat["price_to_sma5"]  = close / sma_5 - 1
    feat["price_to_sma10"] = close / sma_10 - 1
    feat["price_to_sma20"] = close / sma_20 - 1
    feat["price_to_sma50"] = close / sma_50 - 1
    feat["sma20_to_sma50"] = sma_20 / sma_50 - 1

    feat["adx_14"] = ta.trend.ADXIndicator(high, low, close, window=14).adx()

    ret = close.pct_change()
    for window in [5, 10, 20]:
        feat[f"ret_mean_{window}"] = ret.rolling(window).mean()
        feat[f"ret_std_{window}"]  = ret.rolling(window).std()

    # --- Volume ---
    feat["volume_change"] = volume.pct_change()
    feat["volume_sma_ratio"] = volume / volume.rolling(20).mean() - 1

    if prefix:
        feat = feat.add_prefix(prefix)

    return feat

maybank_feat = build_technical_features(maybank_clean)             # e.g. rsi_14, macd, ...
klci_feat    = build_technical_features(klci_clean, prefix="klci_") # e.g. klci_rsi_14, klci_macd, ...

print("Maybank feature columns:", list(maybank_feat.columns))
print("\nKLCI feature columns   :", list(klci_feat.columns))

In [ ]:
# Merge KLCI (market-context) features onto Maybank rows by date, plus a couple of direct
# Maybank-vs-market relative features, which is exactly where the KLCI adds value: relative strength.
features = maybank_feat.join(klci_feat, how="inner")

# Relative-strength features: how Maybank's own return compares to the broader market's return
maybank_ret = maybank_clean["Close"].pct_change()
klci_ret    = klci_clean["Close"].pct_change()
features["maybank_vs_klci_ret_1d"]   = maybank_ret - klci_ret
features["maybank_vs_klci_ret_5d"]   = maybank_ret.rolling(5).mean() - klci_ret.rolling(5).mean()
features["maybank_vs_klci_rsi_diff"] = maybank_feat["rsi_14"] - klci_feat["klci_rsi_14"]

print("Final combined feature matrix shape (pre-NaN-drop):", features.shape)
features.tail()

### Target variable — 3-class trading signal

Instead of a plain up/down label, we define a **3-class trading action** based on the *magnitude* of the next-day return against a configurable threshold band:

- **`1` = Buy**  → next-day return > `+BUY_THRESHOLD`
- **`2` = Sell** → next-day return < `-SELL_THRESHOLD`
- **`0` = Hold** → next-day return falls inside the band (move too small to act on)

As before, this is a forward-looking label, so it's shifted one day relative to the features — at day *t* we only use information available up to and including day *t* to decide the action for day *t+1* (no lookahead bias).

`BUY_THRESHOLD` / `SELL_THRESHOLD` are adjustable: tighter thresholds produce more Buy/Sell rows (more balanced classes, noisier signal), wider thresholds produce more Hold rows (closer to realistic "don't act on noise" behavior).

In [ ]:
BUY_THRESHOLD  = 0.005    # next-day return above +0.5% -> Buy
SELL_THRESHOLD = -0.005   # next-day return below -0.5% -> Sell

LABEL_NAMES = {0: "Hold", 1: "Buy", 2: "Sell"}

next_day_return = maybank_clean["Close"].pct_change().shift(-1)

target = pd.Series(0, index=maybank_clean.index, name="target_signal")  # default: 0 = Hold
target[next_day_return > BUY_THRESHOLD]  = 1   # Buy
target[next_day_return < SELL_THRESHOLD] = 2   # Sell
target[next_day_return.isna()] = np.nan        # last row has no "tomorrow" -> undefined, not Hold

dataset = features.join(target, how="inner")

# Drop rows with NaNs generated by rolling-window warm-up periods (e.g. first 50 rows for SMA-50)
# and the very last row (its target is undefined because there's no "tomorrow" to compare to)
before = len(dataset)
dataset = dataset.dropna()
dataset["target_signal"] = dataset["target_signal"].astype(int)
after = len(dataset)
print(f"Dropped {before - after} warm-up/edge rows with NaNs from indicator rolling windows.")
print(f"Final modeling dataset shape: {dataset.shape}")

class_dist = dataset["target_signal"].value_counts(normalize=True).rename("proportion").sort_index()
class_dist.index = class_dist.index.map(LABEL_NAMES)
class_dist

### Apply the Train / Valid / Test split (from Section 2) to the final feature matrix

We re-derive the split on the **post-NaN-drop** dataset using the same 70/20/10 chronological logic, since dropping warm-up rows shifts the usable date range slightly.

In [ ]:
n_final = len(dataset)
train_end = int(n_final * 0.7)
valid_end = int(n_final * 0.9)

train_df = dataset.iloc[:train_end]
valid_df = dataset.iloc[train_end:valid_end]
test_df  = dataset.iloc[valid_end:]

feature_cols = [c for c in dataset.columns if c != "target_signal"]

X_train, y_train = train_df[feature_cols], train_df["target_signal"]
X_valid, y_valid = valid_df[feature_cols], valid_df["target_signal"]
X_test,  y_test  = test_df[feature_cols],  test_df["target_signal"]

print(f"Train: {X_train.shape}  ({train_df.index.min().date()} -> {train_df.index.max().date()})")
print(f"Valid: {X_valid.shape}  ({valid_df.index.min().date()} -> {valid_df.index.max().date()})")
print(f"Test : {X_test.shape}  ({test_df.index.min().date()} -> {test_df.index.max().date()})  <- production hold-out")

def class_balance_str(y):
    counts = y.value_counts(normalize=True).sort_index()
    return ", ".join(f"{LABEL_NAMES[k]}: {v:.1%}" for k, v in counts.items())

print(f"\nTrain class balance -> {class_balance_str(y_train)}")
print(f"Valid class balance -> {class_balance_str(y_valid)}")
print(f"Test  class balance -> {class_balance_str(y_test)}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train_df.index, maybank_clean.loc[train_df.index, "Close"], label="Train", color="steelblue")
ax.plot(valid_df.index, maybank_clean.loc[valid_df.index, "Close"], label="Valid", color="orange")
ax.plot(test_df.index,  maybank_clean.loc[test_df.index, "Close"],  label="Test (Production)", color="crimson")
ax.set_title("Maybank Close Price — Train / Valid / Test Split (chronological, 7:2:1)")
ax.set_ylabel("Close (MYR)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. XGBoost Hyperparameter Tuning

We tune the `XGBClassifier` with a manual/randomized grid search over the validation set, using:
- `early_stopping_rounds` (fit on train, monitored on valid, stops if valid performance doesn't improve)
- `learning_rate`
- `n_estimators` (upper bound — early stopping finds the actual best number of trees per configuration)
- `max_depth`, `subsample`, `colsample_bytree`, `min_child_weight` for regularization / generalization control

We select the configuration with the **highest validation accuracy**, and record the best number of estimators (boosting rounds) discovered via early stopping.

In [ ]:
from itertools import product

param_grid = {
    "learning_rate":    [0.01, 0.05, 0.1],
    "max_depth":        [3, 4, 5],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
    "min_child_weight": [1, 5],
}

MAX_ESTIMATORS = 1000       # upper ceiling; early stopping will find the real optimum
EARLY_STOPPING_ROUNDS = 30

keys, values = zip(*param_grid.items())
all_combos = [dict(zip(keys, v)) for v in product(*values)]
print(f"Total hyperparameter combinations to try: {len(all_combos)}")

In [ ]:
results_log = []
best_score = -1
best_model = None
best_params = None
best_n_estimators = None

for i, params in enumerate(all_combos):
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=MAX_ESTIMATORS,
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=params["min_child_weight"],
        eval_metric="mlogloss",
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False,
    )

    valid_pred = model.predict(X_valid)
    valid_acc  = accuracy_score(y_valid, valid_pred)
    best_iter  = model.best_iteration if hasattr(model, "best_iteration") else model.n_estimators

    results_log.append({**params, "best_n_estimators": best_iter, "valid_accuracy": valid_acc})

    if valid_acc > best_score:
        best_score = valid_acc
        best_model = model
        best_params = params
        best_n_estimators = best_iter

    if (i + 1) % 10 == 0 or (i + 1) == len(all_combos):
        print(f"[{i+1}/{len(all_combos)}] tried -> current best valid accuracy: {best_score:.4f}")

print("\n" + "="*60)
print("BEST HYPERPARAMETERS FOUND (by validation accuracy)")
print("="*60)
for k, v in best_params.items():
    print(f"  {k:>18}: {v}")
print(f"  {'best_n_estimators':>18}: {best_n_estimators}  (found via early_stopping_rounds={EARLY_STOPPING_ROUNDS})")
print(f"  {'valid_accuracy':>18}: {best_score:.4f}")

In [ ]:
tuning_results_df = pd.DataFrame(results_log).sort_values("valid_accuracy", ascending=False).reset_index(drop=True)
tuning_results_df.head(10)

### Refit the final model

We refit an `XGBClassifier` using the best hyperparameters, with `n_estimators` fixed to the optimal boosting round count found above (rather than relying on early stopping again), trained on Train and validated against Valid one more time for a clean final artifact.

In [ ]:
final_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=best_n_estimators,
    learning_rate=best_params["learning_rate"],
    max_depth=best_params["max_depth"],
    subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    min_child_weight=best_params["min_child_weight"],
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

final_model.fit(X_train, y_train)
print("Final model trained with tuned hyperparameters.")

## 5. Baseline Comparison

With 3 classes, a naive baseline that always predicts a fixed class (e.g. always "Buy") is a sense check for whether the model is doing any better than blindly exploiting whichever class happens to dominate. We use the **majority class observed in the training data** (typically "Hold", since large single-day moves that clear the threshold band are relatively rare) as the baseline prediction for every row.

In [ ]:
majority_class = int(y_train.mode()[0])
majority_class_name = LABEL_NAMES[majority_class]
print(f"Majority class in training data: '{majority_class_name}' ({(y_train == majority_class).mean():.2%} of train rows)")

baseline_predictions = np.full(len(y_test), majority_class, dtype=int)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"\nBaseline (always predict '{majority_class_name}') — Test accuracy: {baseline_accuracy:.4f}")

# Also report baseline on train/valid for context
print(f"Baseline accuracy on Train: {accuracy_score(y_train, np.full(len(y_train), majority_class, dtype=int)):.4f}")
print(f"Baseline accuracy on Valid: {accuracy_score(y_valid, np.full(len(y_valid), majority_class, dtype=int)):.4f}")

## 6. Full Model Performance Results

Comprehensive evaluation across Train, Valid, and Test sets: accuracy, precision, recall, F1, ROC-AUC, confusion matrices, and feature importance.

In [ ]:
CLASS_LABELS = [0, 1, 2]
CLASS_NAMES  = [LABEL_NAMES[c] for c in CLASS_LABELS]  # ["Hold", "Buy", "Sell"]

def evaluate(model, X, y, split_name):
    pred      = model.predict(X)
    pred_prob = model.predict_proba(X)  # shape (n_samples, 3)

    metrics = {
        "split":             split_name,
        "accuracy":          accuracy_score(y, pred),
        "precision_macro":   precision_score(y, pred, average="macro", zero_division=0),
        "recall_macro":      recall_score(y, pred, average="macro", zero_division=0),
        "f1_macro":          f1_score(y, pred, average="macro", zero_division=0),
        "roc_auc_ovr_macro": roc_auc_score(y, pred_prob, multi_class="ovr", average="macro"),
    }
    return metrics, pred, pred_prob

train_metrics, train_pred, train_prob = evaluate(final_model, X_train, y_train, "Train")
valid_metrics, valid_pred, valid_prob = evaluate(final_model, X_valid, y_valid, "Valid")
test_metrics,  test_pred,  test_prob  = evaluate(final_model, X_test,  y_test,  "Test (Production)")

results_summary = pd.DataFrame([train_metrics, valid_metrics, test_metrics]).set_index("split")
results_summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, y_true, pred) in zip(
    axes,
    [("Train", y_train, train_pred), ("Valid", y_valid, valid_pred), ("Test", y_test, test_pred)]
):
    cm = confusion_matrix(y_true, pred, labels=CLASS_LABELS)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{name} Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
print("=== TRAIN classification report ===")
print(classification_report(y_train, train_pred, labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0))

print("=== VALID classification report ===")
print(classification_report(y_valid, valid_pred, labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0))

print("=== TEST (Production) classification report ===")
print(classification_report(y_test, test_pred, labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# Feature importance — which signals (Maybank technicals vs KLCI market-context) matter most
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 8))
sns.barplot(x=importance.values, y=importance.index, palette="viridis")
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

klci_importance_share = importance[[c for c in importance.index if c.startswith("klci_") or "vs_klci" in c]].sum()
print(f"\nShare of total importance coming from FBM KLCI / relative-to-market features: {klci_importance_share / importance.sum():.2%}")

In [ ]:
# One-vs-Rest ROC curves (one per class) on the Test (production) set — with 3 classes there is
# no single ROC curve, so we plot each class against "everything else" and report its own AUC.
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=CLASS_LABELS)
colors = ["#4C72B0", "#55A868", "#C44E52"]

plt.figure(figsize=(6, 6))
for i, cls in enumerate(CLASS_LABELS):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], test_prob[:, i])
    auc = roc_auc_score(y_test_bin[:, i], test_prob[:, i])
    plt.plot(fpr, tpr, label=f"{LABEL_NAMES[cls]} vs. rest (AUC={auc:.3f})", color=colors[i])

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("One-vs-Rest ROC Curves — Test (Production) Set")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Test (Production) Score

The Test split (final 10% chronologically) was **never touched during tuning** — this is our best estimate of how the model would perform if deployed "live" going forward. We compare it directly against the majority-class baseline on the *same* Test set.

In [ ]:
print("="*55)
print("PRODUCTION (TEST) PERFORMANCE SUMMARY")
print("="*55)
print(f"Test period          : {test_df.index.min().date()} -> {test_df.index.max().date()}")
print(f"Number of test days  : {len(y_test)}")
print("-"*55)
print(f"XGBoost Model Accuracy : {test_metrics['accuracy']:.4f}")
print(f"Baseline Accuracy      : {baseline_accuracy:.4f}  (always predicts '{majority_class_name}')")
print(f"Improvement over base. : {test_metrics['accuracy'] - baseline_accuracy:+.4f}")
print("-"*55)
print(f"Precision (macro) : {test_metrics['precision_macro']:.4f}")
print(f"Recall (macro)    : {test_metrics['recall_macro']:.4f}")
print(f"F1-score (macro)  : {test_metrics['f1_macro']:.4f}")
print(f"ROC-AUC (OvR macro): {test_metrics['roc_auc_ovr_macro']:.4f}")
print("="*55)

comparison_df = pd.DataFrame({
    "Model":    ["XGBoost (tuned)", f"Baseline (always {majority_class_name})"],
    "Accuracy": [test_metrics["accuracy"], baseline_accuracy],
})
comparison_df

In [ ]:
plt.figure(figsize=(5, 4))
sns.barplot(data=comparison_df, x="Model", y="Accuracy", palette=["#2E86AB", "#A23B72"])
plt.ylim(0, 1)
plt.axhline(0.5, linestyle="--", color="gray", linewidth=1, label="Random guess (50%)")
plt.xticks(rotation=15)
plt.title("Test (Production) Accuracy: XGBoost vs. Baseline")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Mock Trading Simulation (on Test / Production Data)

With a genuine 3-way **Buy / Hold / Sell** signal, the simulation maps directly onto trading actions for each day in the Test period:

- **Buy (1)**  → go **long** for that day (capture the actual next-day return)
- **Sell (2)** → go **short** for that day (capture the *inverse* of the actual next-day return)
- **Hold (0)** → stay in **cash** for that day (0% return)

> Note: modeling "Sell" as a short position is a simplification for comparison purposes. Short-selling on Bursa Malaysia is restricted to approved securities and incurs borrowing costs/margin requirements not modeled here — in a more conservative simulation, "Sell" could instead just mean "exit to cash," which would produce a more cautious (lower-risk, lower-return) version of this same strategy.

We compare three portfolios on the Test set, all starting from the same capital:
- **XGBoost Model strategy** — acts on the tuned model's daily predictions
- **Baseline strategy** — acts on the majority-class baseline's predictions (Section 5)
- **Buy & Hold** — stays invested in Maybank for the entire test period, ignoring any signal

This is a simplified simulation — it ignores transaction costs, slippage, dividends, taxes, and realistic position sizing, and is meant only to illustrate whether the model's signal has directional value, not to be an actual backtested trading system.

In [ ]:
# Actual next-day returns aligned with the test set
test_actual_returns = maybank_clean["Close"].pct_change().shift(-1).loc[test_df.index]

sim = pd.DataFrame({
    "actual_return":   test_actual_returns,
    "model_signal":    test_pred,
    "baseline_signal": baseline_predictions,
}, index=test_df.index)

def signal_to_return(signal, actual_return):
    """1 = Buy -> long (capture return), 2 = Sell -> short (capture inverse return), 0 = Hold -> cash (0%)."""
    return np.select(
        [signal == 1, signal == 2],
        [actual_return, -actual_return],
        default=0.0,
    )

sim["model_strategy_return"]    = signal_to_return(sim["model_signal"].to_numpy(), sim["actual_return"].to_numpy())
sim["baseline_strategy_return"] = signal_to_return(sim["baseline_signal"].to_numpy(), sim["actual_return"].to_numpy())
sim["buy_hold_return"]          = sim["actual_return"]

STARTING_CAPITAL = 10_000

sim["model_equity"]    = STARTING_CAPITAL * (1 + sim["model_strategy_return"]).cumprod()
sim["baseline_equity"] = STARTING_CAPITAL * (1 + sim["baseline_strategy_return"]).cumprod()
sim["buy_hold_equity"] = STARTING_CAPITAL * (1 + sim["buy_hold_return"]).cumprod()

sim["model_signal_label"] = sim["model_signal"].map(LABEL_NAMES)
print("Model signal distribution on Test set:")
print(sim["model_signal_label"].value_counts())
sim.tail()

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(sim.index, sim["model_equity"],    label="XGBoost Model Strategy", color="#2E86AB", linewidth=2)
plt.plot(sim.index, sim["baseline_equity"], label=f"Baseline (always {majority_class_name})", color="#8C8C8C", linewidth=2, linestyle=":")
plt.plot(sim.index, sim["buy_hold_equity"], label="Buy & Hold", color="#A23B72", linewidth=2, linestyle="--")
plt.axhline(STARTING_CAPITAL, color="gray", linewidth=1, linestyle=":")
plt.title(f"Mock Trading Simulation — Test/Production Period\nStarting Capital: MYR {STARTING_CAPITAL:,}")
plt.ylabel("Portfolio Value (MYR)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def sim_summary(returns, equity, name):
    total_return = equity.iloc[-1] / STARTING_CAPITAL - 1
    n_days = len(returns)
    ann_factor = 252 / n_days
    cagr = (1 + total_return) ** ann_factor - 1
    vol = returns.std() * np.sqrt(252)
    sharpe = (returns.mean() * 252) / (returns.std() * np.sqrt(252) + 1e-9)
    max_dd = (equity / equity.cummax() - 1).min()
    win_rate = (returns > 0).mean()

    return {
        "Strategy": name,
        "Final Equity (MYR)": round(equity.iloc[-1], 2),
        "Total Return": f"{total_return:.2%}",
        "Annualized Return (CAGR)": f"{cagr:.2%}",
        "Annualized Volatility": f"{vol:.2%}",
        "Sharpe Ratio (rf=0)": round(sharpe, 3),
        "Max Drawdown": f"{max_dd:.2%}",
        "Win Rate (days)": f"{win_rate:.2%}",
    }

sim_results = pd.DataFrame([
    sim_summary(sim["model_strategy_return"], sim["model_equity"], "XGBoost Model"),
    sim_summary(sim["baseline_strategy_return"], sim["baseline_equity"], f"Baseline (always {majority_class_name})"),
    sim_summary(sim["buy_hold_return"], sim["buy_hold_equity"], "Buy & Hold"),
]).set_index("Strategy")

sim_results

### Interpretation

- If the **XGBoost Model** strategy shows a higher Sharpe ratio / lower drawdown than Buy & Hold and the baseline, that suggests the model's Buy/Hold/Sell signal (partly informed by FBM KLCI market-context features) has some practical trading value beyond raw accuracy.
- Because "Hold" sits out entirely (0% return, 0% risk) and "Sell" actively bets against the stock, this 3-class strategy can behave very differently from a simple always-invested benchmark — check the **signal distribution** printed above to see how often the model actually chose to act versus stay in cash.
- Remember that **accuracy alone doesn't capture magnitude of moves** — a model can be right more often on small moves and wrong on big ones (or vice versa), and this simulation doesn't weight position size by model confidence.
- Always treat this as a **sanity check**, not investment advice — no transaction costs, slippage, dividends, margin/borrowing costs for shorting, or realistic position sizing were modeled, and past performance on this single stock/period is not a guarantee of future results.